In [ ]:
import torch

In [ ]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            if v.grad_fn is not None:
                for child, _ in v.grad_fn.next_functions:
                    if child is not None:
                        edges.add((child, v.grad_fn))
                        build_fn(child)
    def build_fn(fn):
        if fn not in nodes:
            nodes.add(fn)
            for child, _ in fn.next_functions:
                if child is not None:
                    edges.add((child, fn))
                    build_fn(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir})
    
    for n in nodes:
        if isinstance(n, torch.Tensor):
            label = "{ data %.4f | grad %.4f }" % (n.data.item(), n.grad.item() if n.grad is not None else 0.0)
            dot.node(name=str(id(n)), label=label, shape='record')
        else:
            # This is a grad_fn (operation node)
            op_name = type(n).__name__.replace('Backward0', '').replace('Backward1', '')
            dot.node(name=str(id(n)), label=op_name)
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)))
    
    # Connect tensor to its grad_fn
    if root.grad_fn is not None:
        dot.edge(str(id(root.grad_fn)), str(id(root)))
    
    return dot

In [ ]:
# a very simple example
x = torch.tensor(1.0, requires_grad=True)
y = (x * 2 + 1).relu()
y.backward()
draw_dot(y)

In [ ]:
# a simple 2D neuron
import random
import torch.nn as nn

torch.manual_seed(1337)
n = nn.Linear(2, 1)
x = torch.tensor([[1.0, -2.0]], requires_grad=True)
y = n(x).relu()
y.backward()

dot = draw_dot(y.squeeze())
dot

In [ ]:
# a simple 3D neuron
import random
import torch.nn as nn

torch.manual_seed(1337)
n = nn.Linear(3, 1)
x = torch.tensor([[1.0, -2.0, 3.0]], requires_grad=True)
y = n(x).relu()
y.backward()

dot = draw_dot(y.squeeze())
dot

In [ ]:
dot.render('gout_torch')